In [3]:
# ============================================================
#  ewl_multi_nash.py  •  grid‑vs‑optimiser Nash search (EWL)
#  symmetric entangler – simpler binding (no compose bug)
# ============================================================
import itertools, math, sys, time
from collections import defaultdict

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit_aer import AerSimulator

try:
    from scipy.optimize import minimize
    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False
    print("SciPy not found – optimiser mode will be skipped.", file=sys.stderr)

# ------------------- game / circuit settings ----------------------
T, R, P, S = 5, 3, 1, 0
GAMMA      = math.pi / 2
BACKEND    = AerSimulator(method="statevector")

# ------------ angle‑grid resolution ------------------------------
GRID_RES_THETA = 15   # θ grid points
GRID_RES_PHI   = 8    # φ grid points

theta_grid = np.linspace(0, math.pi,  GRID_RES_THETA)
phi_grid   = np.linspace(0, math.pi/2, GRID_RES_PHI)
GRID       = list(itertools.product(theta_grid, phi_grid))

# ============================================================
#  Gates
# ============================================================

def entangler(n: int, gamma: float):
    qc = QuantumCircuit(n, name="J")
    for j in range(n):
        for k in range(j+1, n):
            qc.cx(j, k)
            qc.rz(2*gamma, k)
            qc.cx(j, k)
    return qc.to_gate()


def payoff_vector(bits: str):
    n = len(bits)
    d = bits.count("1")
    if d == 0:
        return [R]*n
    if d == n:
        return [P]*n
    return [T if b=="1" else S for b in bits]

# ============================================================
#  Parameterised circuit builder
# ============================================================

def build_parameterised_ewl(n):
    theta = [Parameter(f"θ{q}") for q in range(n)]
    phi   = [Parameter(f"φ{q}") for q in range(n)]
    qc = QuantumCircuit(n)
    qc.append(entangler(n, GAMMA), range(n))
    for q in range(n):
        qc.ry(theta[q], q)
        qc.rz(phi[q],   q)
    qc.append(entangler(n, -GAMMA), range(n))
    qc.save_statevector()
    return qc, theta, phi

# ============================================================
#  GRID‑BASED Nash search (straightforward version)
# ============================================================

def ewl_grid_nash(n_players):
    t0 = time.perf_counter()
    base, THETA_PAR, PHI_PAR = build_parameterised_ewl(n_players)
    tqc = transpile(base, BACKEND)
    t_transpile = time.perf_counter() - t0

    profile_payoff = {}
    t1 = time.perf_counter()
    for profile in itertools.product(range(len(GRID)), repeat=n_players):
        bind = {}
        for q, idx in enumerate(profile):
            θ, φ = GRID[idx]
            bind[THETA_PAR[q]] = θ
            bind[PHI_PAR[q]]   = φ
        vec   = BACKEND.run(tqc.assign_parameters(bind, inplace=False)).result().get_statevector()
        probs = np.abs(vec)**2
        pay   = np.zeros(n_players)
        for k, p in enumerate(probs):
            if p == 0.0:
                continue
            bits = format(k, f"0{n_players}b")
            pay += p*np.array(payoff_vector(bits))
        profile_payoff[profile] = tuple(pay.tolist())
    t_sim = time.perf_counter() - t1

    best_val   = defaultdict(lambda: float('-inf'))
    best_strat = {}
    for prof, pay in profile_payoff.items():
        for p in range(n_players):
            opp = prof[:p]+prof[p+1:]
            key = (p, opp)
            if pay[p] > best_val[key]:
                best_val[key] = pay[p]
                best_strat[key] = prof[p]
    nash = [(prof, pay) for prof, pay in profile_payoff.items()
            if all(prof[p]==best_strat[(p, prof[:p]+prof[p+1:])] for p in range(n_players))]

    timing = dict(transpile=t_transpile,
                  simulation=t_sim,
                  total=time.perf_counter()-t0)
    return nash, timing, profile_payoff

# ============================================================
#  Continuous optimiser (optional, unchanged)
# ============================================================

def ewl_local_opt_nash(n_players, start_angles):
    if not HAVE_SCIPY:
        raise RuntimeError("SciPy not installed")

    base, THETA_PAR, PHI_PAR = build_parameterised_ewl(n_players)
    tqc = transpile(base, BACKEND)

    def expected_payoff(angle_vec):
        bind = {}
        for q in range(n_players):
            θ, φ = angle_vec[2*q:2*q+2]
            bind[THETA_PAR[q]] = θ
            bind[PHI_PAR[q]]   = φ
        vec = BACKEND.run(tqc.assign_parameters(bind, inplace=False)).result().get_statevector()
        probs = np.abs(vec)**2
        pay = np.zeros(n_players)
        for k, p in enumerate(probs):
            if p==0.0:
                continue
            bits=format(k,f"0{n_players}b")
            pay += p*np.array(payoff_vector(bits))
        return pay

    def player_best_response(player, others):
        def obj(x):
            ang = others.copy()
            ang[2*player:2*player+2] = x
            return -expected_payoff(ang)[player]
        res = minimize(obj, x0=others[2*player:2*player+2],
                       bounds=[(0, math.pi), (0, math.pi/2)],
                       method="Nelder-Mead",
                       options=dict(maxiter=200, fatol=1e-4))
        return res.x

    angles = np.array(start_angles, dtype=float)
    for _ in range(50):
        prev = angles.copy()
        for p in range(n_players):
            angles[2*p:2*p+2] = player_best_response(p, angles)
        if np.allclose(prev, angles, atol=1e-3):
            break
    return angles, expected_payoff(angles)

# ============================================================
#  DEMO
# ============================================================
if __name__ == "__main__":
    for n in (2,3):
        print(f"\n===== GRID search  n = {n}  =====")
        nash, timing, prof_pay = ewl_grid_nash(n)
        if nash:
            for prof, pay in nash:
                angs=[GRID[i] for i in prof]
                print(" idx", prof, "→ angles", angs, "→ pay-offs", tuple(round(x,3) for x in pay))
        else:
            print("   No pure-strategy NE on this grid.")
        print(" timing:", timing)

    if HAVE_SCIPY:
        print("\n===== Continuous optimiser demo  n = 3  =====")
        start=[0,0, math.pi,0, 0,0]
        opt_ang,opt_pay = ewl_local_opt_nash(3,start)
        print(" converged angles:",[(opt_ang[2*q],opt_ang[2*q+1]) for q in range(3)])
        print(" pay-offs:", tuple(round(x,3) for x in opt_pay))



===== GRID search  n = 2  =====
 idx (0, 0) → angles [(0.0, 0.0), (0.0, 0.0)] → pay-offs (3.0, 3.0)
 idx (1, 6) → angles [(0.0, 0.2243994752564138), (0.0, 1.3463968515384828)] → pay-offs (3.0, 3.0)
 idx (3, 3) → angles [(0.0, 0.6731984257692414), (0.0, 0.6731984257692414)] → pay-offs (3.0, 3.0)
 timing: {'transpile': 0.11131347808986902, 'simulation': 11.656691580079496, 'total': 11.78982113674283}

===== GRID search  n = 3  =====
 idx (0, 112, 4) → angles [(0.0, 0.0), (3.141592653589793, 0.0), (0.0, 0.8975979010256552)] → pay-offs (0.0, 5.0, 0.0)
 idx (0, 114, 1) → angles [(0.0, 0.0), (3.141592653589793, 0.4487989505128276), (0.0, 0.2243994752564138)] → pay-offs (0.0, 5.0, 0.0)
 idx (2, 112, 1) → angles [(0.0, 0.4487989505128276), (3.141592653589793, 0.0), (0.0, 0.2243994752564138)] → pay-offs (0.0, 5.0, 0.0)
 idx (2, 113, 0) → angles [(0.0, 0.4487989505128276), (3.141592653589793, 0.2243994752564138), (0.0, 0.0)] → pay-offs (0.0, 5.0, 0.0)
 idx (3, 114, 2) → angles [(0.0, 0.67319842